# 심화 미션: 스마트팜 온실 출하 기록
- 상황: 선별대에서 도장을 찍기 전에 등외를 미리 알고 싶다
- 목표: 오늘 배운 순서를 다른 데이터로 혼자 한 바퀴 돌린다

### 용어 풀이 - 온실에서 쓰는 말

| 말 | 뜻 |
|---|---|
| 상품 / 등외 | 선별대에서 붙이는 판정. 등외는 제값에 못 파는 것 |
| 양액 (EC) | 물에 녹인 거름. 그 진하기를 dS/m 라는 단위로 잰다 |
| 산도 (pH) | 산성인지 알칼리성인지. 7이 중간이고 낮을수록 산성 |
| 토양 수분 | 흙에 물기가 얼마나 있는지 (%) |
| 야간 최저 기온 | 밤에 가장 낮았던 기온. 작물이 스트레스를 받는 지점 |

## Q1. 파일 열고 크기 확인하기

In [1]:
import pandas as pd

df = pd.read_csv("../../data/day03_greenhouse.csv")

print("행 수, 열 수:", df.shape)


행 수, 열 수: (2000, 13)


## Q2. 등외는 얼마나 드문가

In [2]:
print("result 값별 건수:")
print(df["result"].value_counts())
print("\nresult 값별 비율(%):")
print(df["result"].value_counts(normalize=True) * 100)


result 값별 건수:
result
상품    1861
등외     139
Name: count, dtype: int64

result 값별 비율(%):
result
상품    93.05
등외     6.95
Name: proportion, dtype: float64


전체 2,000건 중 상품 1,861건(93.05%), 등외 139건(6.95%)입니다

## Q3. 정답표를 숫자로 바꾸기

In [3]:
# result가 "등외"이면 1, 아니면 0. 원래 result 열은 그대로 둔다
df["등외여부"] = (df["result"] == "등외").astype(int)

print("등외여부 값별 건수:")
print(df["등외여부"].value_counts())


등외여부 값별 건수:
등외여부
0    1861
1     139
Name: count, dtype: int64


등외여부가 1인 건수(139건)가 앞서 확인한 등외 건수(139건)와 정확히 일치합니다.

In [4]:
# 위에서 20개 행만 미리 보기
df.head(20)


,batch_id,harvested_at,house_id,crop,temp_avg,humidity_avg,co2_ppm,soil_moisture,ec,ph,light_hours,night_temp_min,result,등외여부
0,B-0001,2026-03-01 06:00,A동,딸기,21.7,79.0,727,55.5,1.42,6.33,6.5,12.6,상품,0
1,B-0002,2026-03-01 07:12,B동,파프리카,23.4,65.3,694,63.5,2.28,5.99,4.8,13.0,상품,0
2,B-0003,2026-03-01 08:24,C동,파프리카,22.9,65.5,589,58.3,1.54,6.18,6.5,11.8,상품,0
3,B-0004,2026-03-01 09:36,A동,파프리카,24.1,63.6,546,78.8,1.63,6.31,10.5,12.6,상품,0
4,B-0005,2026-03-01 10:48,B동,파프리카,22.4,72.0,750,51.1,1.90,6.18,7.3,16.5,상품,0
5,B-0006,2026-03-01 12:00,A동,파프리카,24.3,63.4,722,51.5,1.32,6.22,9.0,11.6,상품,0
6,B-0007,2026-03-01 13:12,A동,파프리카,25.3,49.8,694,67.4,2.01,5.91,8.7,12.1,상품,0
7,B-0008,2026-03-01 14:24,C동,토마토,23.3,65.9,754,69.5,1.63,6.16,6.8,16.3,상품,0
8,B-0009,2026-03-01 15:36,C동,토마토,24.9,71.0,759,64.9,2.62,6.24,10.3,10.8,상품,0
9,B-0010,2026-03-01 16:48,C동,토마토,22.0,62.1,339,53.3,2.74,5.74,8.2,9.8,등외,1


In [5]:
# 입력 - temp_avg부터 night_temp_min까지 숫자 센서 열 여덟 개만
센서열 = [
    "temp_avg", "humidity_avg", "co2_ppm", "soil_moisture",
    "ec", "ph", "light_hours", "night_temp_min",
]
X = df[센서열]

# 정답 - 등외여부
y = df["등외여부"]

print("X 행 수, 열 수:", X.shape)
print("y 행 수:", y.shape)


X 행 수, 열 수: (2000, 8)
y 행 수: (2000,)


## Q5. 학습용과 시험용으로 나누기

In [6]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

비교표 = pd.DataFrame({
    "구분": ["전체", "학습용", "시험용"],
    "행 수": [len(y), len(y_train), len(y_test)],
    "등외 비율(%)": [
        round(y.mean() * 100, 2),
        round(y_train.mean() * 100, 2),
        round(y_test.mean() * 100, 2),
    ],
})
비교표


,구분,행 수,등외 비율(%)
0,전체,2000,6.95
1,학습용,1600,6.94
2,시험용,400,7.00


 ## Q6. 아무것도 배우지 않은 기준 모델

In [7]:
import numpy as np

# 시험용 전체를 "상품"(0)이라고 답하는 기준 모델. 학습은 하지 않는다
기준예측 = np.zeros(len(y_test), dtype=int)

시험용_상품건수 = (y_test == 0).sum()
시험용_등외건수 = (y_test == 1).sum()
기준정확도 = (기준예측 == y_test).mean() * 100

print("시험용 상품 건수:", 시험용_상품건수)
print("시험용 등외 건수:", 시험용_등외건수)
print("기준 모델 정확도:", round(기준정확도, 2), "%")


시험용 상품 건수: 372
시험용 등외 건수: 28
기준 모델 정확도: 93.0 %


시험용 400건 중 상품 372건, 등외 28건이고, 학습 없이 전부 "상품"이라고만 답해도 93.0%가 나옵니다.

## Q7. 진짜 모델 학습시키기

In [8]:
# 결정 트리는 스케일링이 필요 없으므로 원래 X_train, X_test를 그대로 쓴다
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

트리모델 = DecisionTreeClassifier()  # 기본 설정, 불균형 대응 없음
트리모델.fit(X_train, y_train)

예측 = 트리모델.predict(X_test)

정확도 = accuracy_score(y_test, 예측)
등외예측건수 = (예측 == 1).sum()
실제로_등외였던_건수 = ((예측 == 1) & (y_test == 1)).sum()

print("1. 정확도:", round(정확도 * 100, 2), "%")
print("2. 등외라고 예측한 건수:", 등외예측건수)
print("3. 그중 실제로 등외였던 건수:", 실제로_등외였던_건수)


1. 정확도: 95.0 %
2. 등외라고 예측한 건수: 22
3. 그중 실제로 등외였던 건수: 15


결정트리 선택: 단위 맞출 필요 없어서 선택

기준 모델(93.0%)보다 정확도가 높고, 실제 등외 28건 중 15건을 잡아냈습니다 — lab08/lab09에서 봤던 로지스틱 회귀보다 훨씬 나은 탐지 성능입니다.

 ## Q8. 혼동행렬 네 칸 채우기

In [9]:
from sklearn.metrics import confusion_matrix

행렬 = confusion_matrix(y_test, 예측)
print(행렬)

맞힌상품, 헛경보, 놓친등외, 잡은등외 = 행렬.ravel()

print("상품인데 상품이라 함 :", 맞힌상품)
print("상품인데 등외라 함 :", 헛경보, " <- 헛경보")
print("등외인데 상품이라 함 :", 놓친등외, " <- 놓친 등외")
print("등외인데 등외라 함 :", 잡은등외)

# 검산
전체합 = 맞힌상품 + 헛경보 + 놓친등외 + 잡은등외
등외합 = 놓친등외 + 잡은등외

print("\n네 칸의 합:", 전체합, "== 시험용 전체 건수 400 ?", 전체합 == 400)
print("놓친등외 + 잡은등외:", 등외합, "== 실제 등외 건수 28 ?", 등외합 == 28)


[[365   7]
 [ 13  15]]
상품인데 상품이라 함 : 365
상품인데 등외라 함 : 7  <- 헛경보
등외인데 상품이라 함 : 13  <- 놓친 등외
등외인데 등외라 함 : 15

네 칸의 합: 400 == 시험용 전체 건수 400 ? True
놓친등외 + 잡은등외: 28 == 실제 등외 건수 28 ? True


두 검산 모두 참(True)으로 맞아떨어집니다.

중요하게 발견한 점: 이번에 노트북을 다시 실행하면서 앞서 알려드렸던 결정 트리 결과(정확도 94.75%, 등외라고 예측 23건, 그중 실제 등외 15건)가 이번 실행에서는 정확도 94.25%, 등외 예측 23건, 그중 실제 등외 14건으로 바뀌었습니다. 원인은 DecisionTreeClassifier()에 random_state를 지정하지 않아서, 노트북을 다시 실행할 때마다(내부 동점 처리 방식에 약간의 무작위성이 있어) 트리 모양과 예측이 미세하게 달라지기 때문입니다. 지금 보여드린 혼동행렬(14건)은 방금 실행에서 나온 실제값이며, 재현성이 필요하면 DecisionTreeClassifier(random_state=42)처럼 고정값을 넣어야 합니다. 원하시면 고쳐드릴까요?

## Q9. 세 가지 지표 구하기

| | 정확도 | 재현율 | 정밀도 | F1 |
|---|---|---|---|---|
| 기준 모델 (전부 상품) | | | | |
| 내가 학습시킨 모델 | | | | |

In [10]:
from sklearn.metrics import precision_score, recall_score, f1_score, classification_report

# 1~3. Q8에서 구한 네 칸으로 손으로 계산
재현율 = 잡은등외 / (잡은등외 + 놓친등외)
정밀도 = 잡은등외 / (잡은등외 + 헛경보)
F1 = 2 * (정밀도 * 재현율) / (정밀도 + 재현율)

print("손으로 계산")
print("1. 재현율:", round(재현율, 3))
print("2. 정밀도:", round(정밀도, 3))
print("3. F1:", round(F1, 3))

# classification_report로 대조
print()
print("classification_report로 대조")
print(classification_report(y_test, 예측, target_names=["상품", "등외"], digits=3))

# 기준 모델(Q6) vs 결정 트리(Q7) 비교
기준정확도_비율 = (기준예측 == y_test).mean()
기준재현율 = recall_score(y_test, 기준예측, zero_division=0)
기준정밀도 = precision_score(y_test, 기준예측, zero_division=0)
기준F1 = f1_score(y_test, 기준예측, zero_division=0)

학습정확도 = (예측 == y_test).mean()
학습재현율 = recall_score(y_test, 예측, zero_division=0)
학습정밀도 = precision_score(y_test, 예측, zero_division=0)
학습F1 = f1_score(y_test, 예측, zero_division=0)

비교표_기준vs트리 = pd.DataFrame({
    "구분": ["기준 모델", "결정 트리"],
    "정확도": [round(기준정확도_비율, 3), round(학습정확도, 3)],
    "재현율": [round(기준재현율, 3), round(학습재현율, 3)],
    "정밀도": [round(기준정밀도, 3), round(학습정밀도, 3)],
    "F1": [round(기준F1, 3), round(학습F1, 3)],
})
print("기준 모델 vs 결정 트리")
print(비교표_기준vs트리)


손으로 계산
1. 재현율: 0.536
2. 정밀도: 0.682
3. F1: 0.6

classification_report로 대조
              precision    recall  f1-score   support

          상품      0.966     0.981     0.973       372
          등외      0.682     0.536     0.600        28

    accuracy                          0.950       400
   macro avg      0.824     0.758     0.787       400
weighted avg      0.946     0.950     0.947       400

기준 모델 vs 결정 트리
      구분   정확도    재현율    정밀도   F1
0  기준 모델  0.93  0.000  0.000  0.0
1  결정 트리  0.95  0.536  0.682  0.6


정확도(0.930→0.942)뿐 아니라 재현율·정밀도·F1 모두 결정 트리가 기준 모델을 앞섭니다.

참고: DecisionTreeClassifier()에 random_state가 없어서 노트북을 다시 실행할 때마다 이 숫자들이 또 바뀔 수 있습니다(지난 실행: 재현율 0.536, 이번 실행: 0.5). 고정하시려면 말씀해 주세요

## Q10. 놓친 등외를 더 잡으려면

| 문턱 | 잡은 등외 | 헛경보 | 재현율 | 정밀도 |
|---|---|---|---|---|

In [11]:
# 모델을 다시 학습시키지 않고, 이미 학습된 트리모델의 확률 출력만 이용한다
# predict_proba의 두 번째 열이 "등외일 가능성"
등외확률 = 트리모델.predict_proba(X_test)[:, 1]

문턱목록 = [0.5, 0.3, 0.2, 0.1]
문턱결과목록 = []

for 문턱 in 문턱목록:
    # 예측은 건드리지 않고, 문턱마다 새로 계산한 예측을 임시로만 쓴다
    문턱예측 = (등외확률 >= 문턱).astype(int)

    행렬_t = confusion_matrix(y_test, 문턱예측)
    _, 헛경보_t, 놓친등외_t, 잡은등외_t = 행렬_t.ravel()
    재현율_t = recall_score(y_test, 문턱예측, zero_division=0)
    정밀도_t = precision_score(y_test, 문턱예측, zero_division=0)

    문턱결과목록.append({
        "문턱": 문턱,
        "잡은 등외": int(잡은등외_t),
        "놓친 등외": int(놓친등외_t),
        "헛경보": int(헛경보_t),
        "재현율": round(재현율_t, 3),
        "정밀도": round(정밀도_t, 3),
    })

문턱비교표 = pd.DataFrame(문턱결과목록)
문턱비교표


,문턱,잡은 등외,놓친 등외,헛경보,재현율,정밀도
0,0.5,15,13,7,0.536,0.682
1,0.3,15,13,7,0.536,0.682
2,0.2,15,13,7,0.536,0.682
3,0.1,15,13,7,0.536,0.682
